# PoC 2d: Hypothesis-First Foresight

Invert the poc2c pipeline: instead of extracting what the corpus contains, generate disruption hypotheses and then search for evidence to validate or falsify them.

**Domain:** Regulatory Spectrum Monitoring (R&S core business)

**Pipeline:** LLM Hypotheses (5 frameworks) → OpenAlex Evidence Search → Faithfulness Check → Categorize (Validated / Speculative / Falsified) → Surprise Assessment → Audit Trail

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import AzureOpenAI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path

from src.models import (
    Hypothesis, HypothesisBatch, HypothesisVerdict,
    FaithfulnessResult, SurpriseAssessment,
)

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")

print(f"LLM: {MODEL}")

## 1. Domain Context: Regulatory Spectrum Monitoring

In [ ]:
DOMAIN_CONTEXT = """DOMAIN: Regulatory Spectrum Monitoring

Rohde & Schwarz (R&S) is a leading provider of spectrum monitoring solutions for national
regulatory authorities and defense agencies worldwide. Their portfolio includes:

- Fixed monitoring stations and nationwide monitoring networks
- Mobile and portable spectrum monitoring receivers
- Direction finding and geolocation systems (DF/TDOA)
- Automated signal identification and classification
- Spectrum management databases and software platforms

Current status quo:
- Hardware-centric business: high-performance RF receivers, antennas, DF arrays
- Customers: national regulators (BNetzA, FCC, Ofcom, etc.), military/intelligence agencies
- Market drivers: growing spectrum demand (5G/6G, IoT, satellite), interference management
- Known investments: AI-assisted signal classification, wider bandwidth receivers, cloud-based monitoring

What R&S already tracks (known trends):
- 5G/6G spectrum allocation and testing
- Dynamic Spectrum Access / CBRS-style sharing
- Satellite mega-constellations (Starlink, etc.) and spectrum coordination
- AI/ML for automated signal recognition
- Software-defined receivers and virtualized monitoring
- Electronic warfare / SIGINT modernization
"""

print(DOMAIN_CONTEXT)

## 2. Hypothesis Generation (5 Disruption Frameworks)

One LLM call per framework — 6 hypotheses each = 30 total. Temperature 0.7 for creative exploration.

In [ ]:
FRAMEWORKS = {
    "Christensen Disruption": (
        "Identify technologies that are currently 'good enough' and cheap, coming from below "
        "to undermine R&S's high-performance spectrum monitoring hardware. Think: what if a "
        "$50 SDR dongle network replaces a $500k monitoring station? What low-end approaches "
        "from adjacent fields could become 'good enough' for regulatory monitoring?"
    ),
    "Adjacent Possible": (
        "Identify combinations of existing technologies that are JUST NOW becoming feasible "
        "and could create entirely new approaches to spectrum monitoring. Think: what happens "
        "when LEO satellites + edge AI + crowdsourced sensors converge? What new capabilities "
        "emerge from combining mature technologies in novel ways?"
    ),
    "Regulatory Shock": (
        "Identify policy or regulatory changes that could fundamentally restructure how "
        "spectrum is monitored, allocated, or enforced. Think: what if spectrum becomes "
        "a real-time market? What if monitoring obligations shift from regulators to operators? "
        "What geopolitical shifts could fragment global spectrum governance?"
    ),
    "Platform Shift": (
        "Identify shifts in where value is created in the spectrum monitoring chain. Think: "
        "what if the value moves from receivers to analytics platforms? What if monitoring "
        "becomes a cloud service instead of a hardware deployment? Who could become the "
        "new platform owner that R&S depends on or competes with?"
    ),
    "Convergence": (
        "Identify industries or domains that are merging with spectrum monitoring in unexpected "
        "ways. Think: cybersecurity + spectrum = RF threat detection? Autonomous vehicles + "
        "spectrum = mobile sensing platforms? Climate monitoring + spectrum = dual-use infrastructure? "
        "What cross-industry convergences create new competitors or opportunities?"
    ),
}

HYPOTHESIS_SYSTEM = f"""You are a strategic foresight analyst specializing in technology disruption.
Your task: generate non-obvious, specific disruption hypotheses for a given domain using a
specific disruption framework.

{DOMAIN_CONTEXT}

RULES:
- Each hypothesis must be SPECIFIC and FALSIFIABLE, not vague ("AI will change things")
- Focus on what R&S is NOT already tracking (see known trends above)
- Think 5-10 year horizon
- For each hypothesis, generate 2-4 concrete search queries that could find supporting evidence
- Prefer cross-domain, second-order, and non-obvious angles
- Number hypotheses sequentially within the batch (the id will be set by the framework prefix)
"""

all_hypotheses: list[Hypothesis] = []

for fw_name, fw_prompt in FRAMEWORKS.items():
    prefix = fw_name[:3].upper()  # CHR, ADJ, REG, PLA, CON

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": HYPOTHESIS_SYSTEM},
            {"role": "user", "content": (
                f"Framework: {fw_name}\n\n"
                f"{fw_prompt}\n\n"
                f"Generate exactly 6 disruption hypotheses. Use IDs {prefix}-01 through {prefix}-06."
            )},
        ],
        response_format=HypothesisBatch,
        temperature=0.7,
    )

    batch = response.choices[0].message.parsed.hypotheses
    all_hypotheses.extend(batch)
    print(f"\n{fw_name} ({len(batch)} hypotheses):")
    for h in batch:
        print(f"  {h.id}: {h.hypothesis[:100]}...")
    time.sleep(1)

print(f"\nTotal: {len(all_hypotheses)} hypotheses across {len(FRAMEWORKS)} frameworks")

In [ ]:
hyp_df = pd.DataFrame([
    {"ID": h.id, "Framework": h.disruption_framework, "Hypothesis": h.hypothesis[:120],
     "Queries": len(h.search_queries), "R&S Relevance": h.relevance_to_rs[:80]}
    for h in all_hypotheses
])
hyp_df

## 3. Evidence Search (OpenAlex)

For each hypothesis, run its generated search queries against OpenAlex. "No results" is a valid signal.

In [ ]:
OA_BASE = "https://api.openalex.org"
OA_PARAMS = {"mailto": "paul.keck@campus.lmu.de"}

def oa_reconstruct_abstract(inv_index: dict | None) -> str | None:
    if not inv_index:
        return None
    words = {}
    for word, positions in inv_index.items():
        for pos in positions:
            words[pos] = word
    return " ".join(words[i] for i in sorted(words))

def oa_search(query: str, limit: int = 5) -> list[dict]:
    try:
        resp = requests.get(
            f"{OA_BASE}/works",
            params={**OA_PARAMS, "search": query, "per_page": limit},
            timeout=10,
        )
        resp.raise_for_status()
        results = []
        for w in resp.json().get("results", []):
            abstract = oa_reconstruct_abstract(w.get("abstract_inverted_index"))
            if abstract:
                results.append({
                    "openalex_id": w["id"],
                    "title": w.get("display_name", ""),
                    "abstract": abstract,
                    "year": w.get("publication_year"),
                    "cited_by_count": w.get("cited_by_count", 0),
                })
        return results
    except Exception as e:
        print(f"  Search failed for '{query[:50]}': {e}")
        return []

PAPERS_PER_QUERY = 5

hypothesis_evidence: dict[str, list[dict]] = {}

for h in all_hypotheses:
    papers = []
    seen_ids = set()
    for query in h.search_queries:
        results = oa_search(query, limit=PAPERS_PER_QUERY)
        for p in results:
            if p["openalex_id"] not in seen_ids:
                seen_ids.add(p["openalex_id"])
                papers.append({**p, "query": query})
        time.sleep(0.3)

    hypothesis_evidence[h.id] = papers
    status = f"{len(papers)} papers" if papers else "NO EVIDENCE"
    print(f"  {h.id}: {status}")

total_papers = sum(len(ps) for ps in hypothesis_evidence.values())
empty = sum(1 for ps in hypothesis_evidence.values() if not ps)
print(f"\nTotal: {total_papers} papers for {len(all_hypotheses)} hypotheses ({empty} with no evidence)")

## 4. Faithfulness Check

For each hypothesis that has evidence: decompose into atomic claims, verify against paper abstracts. Reuses the poc2c pattern.

In [ ]:
FAITHFULNESS_THRESHOLD = 0.5

faithfulness_results: dict[str, FaithfulnessResult] = {}

for h in all_hypotheses:
    papers = hypothesis_evidence[h.id]
    if not papers:
        continue

    evidence_block = "\n\n".join(
        f'[Source: "{p["title"]}" ({p["openalex_id"]})]{chr(10)}{p["abstract"]}'
        for p in papers[:8]
    )

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a fact-checking assistant. Given a disruption hypothesis and academic "
                "paper abstracts, decompose the hypothesis into atomic factual claims and verify "
                "each against the evidence.\n\n"
                "For each claim:\n"
                "- If DIRECTLY supported by a source passage, mark supported=true and quote it\n"
                "- If NOT supported (extrapolation, opinion, or no matching passage), mark supported=false\n\n"
                "Be strict: only mark a claim as supported if the source text clearly states or "
                "strongly implies it. The hypothesis may be partially supported — that is fine."
            )},
            {"role": "user", "content": (
                f"HYPOTHESIS ({h.id}): {h.hypothesis}\n\n"
                f"EXPECTED EVIDENCE: {h.expected_evidence_type}\n\n"
                f"SOURCE EVIDENCE:\n{evidence_block}"
            )},
        ],
        response_format=FaithfulnessResult,
        temperature=0.1,
    )

    result = response.choices[0].message.parsed
    faithfulness_results[h.id] = result
    status = "PASS" if result.faithfulness_score >= FAITHFULNESS_THRESHOLD else "FAIL"
    print(f"  {h.id}: {result.faithfulness_score:.0%} [{status}] — {len(result.claims)} claims")
    time.sleep(0.5)

print(f"\nChecked {len(faithfulness_results)}/{len(all_hypotheses)} hypotheses (rest had no evidence)")

## 5. Categorize: Validated / Speculative / Falsified

In [ ]:
validated, speculative, falsified = [], [], []

for h in all_hypotheses:
    papers = hypothesis_evidence[h.id]
    faith = faithfulness_results.get(h.id)

    if not papers:
        verdict = "speculative"
        summary = "No academic evidence found. Hypothesis is LLM-generated without external support."
    elif faith and faith.faithfulness_score >= FAITHFULNESS_THRESHOLD:
        verdict = "validated"
        supported = [c for c in faith.claims if c.supported]
        summary = f"{len(supported)}/{len(faith.claims)} claims supported. {faith.unsupported_claims_summary}"
    elif faith and faith.faithfulness_score < 0.2:
        verdict = "falsified"
        summary = f"Evidence found but contradicts or does not support hypothesis. Score: {faith.faithfulness_score:.0%}"
    else:
        verdict = "speculative"
        score_str = f"{faith.faithfulness_score:.0%}" if faith else "N/A"
        summary = f"Some evidence but weak support (faithfulness: {score_str}). Treat as unvalidated hypothesis."

    hv = HypothesisVerdict(
        hypothesis_id=h.id,
        hypothesis=h.hypothesis,
        verdict=verdict,
        faithfulness_result=faith,
        evidence_summary=summary,
        papers_found=len(papers),
        search_queries_used=h.search_queries,
    )

    if verdict == "validated":
        validated.append(hv)
    elif verdict == "falsified":
        falsified.append(hv)
    else:
        speculative.append(hv)

print(f"Validated:   {len(validated)}")
print(f"Speculative: {len(speculative)}")
print(f"Falsified:   {len(falsified)}")
print(f"Total:       {len(all_hypotheses)}")

## 6. Surprise Assessment

For validated and speculative hypotheses: how surprising would this be to an R&S strategist?

In [ ]:
RS_MONITORING_CONTEXT = """Rohde & Schwarz Spectrum Monitoring context:
- World leader in regulatory spectrum monitoring systems
- Core products: fixed monitoring stations, mobile receivers, DF/geolocation, automated signal ID
- Customers: national regulators (BNetzA, FCC, Ofcom, ANFR, etc.), defense/intelligence agencies
- Known priorities: AI signal classification, wider bandwidth, cloud-based monitoring, 5G/6G readiness
- Business model: high-end hardware + software platforms, long procurement cycles (government)
- Revenue ~EUR 3B total (monitoring is a division), Munich-based, privately held
- Competitive landscape: Keysight, Anritsu, PCTEL, TCI, smaller players"""

surprise_results: dict[str, SurpriseAssessment] = {}

for hv in validated + speculative:
    evidence_note = (
        f"Evidence: {hv.papers_found} papers, faithfulness {hv.faithfulness_result.faithfulness_score:.0%}"
        if hv.faithfulness_result
        else "Evidence: NONE (speculative hypothesis)"
    )

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You assess disruption hypotheses for blind spots in R&S strategic planning.\n\n"
                f"{RS_MONITORING_CONTEXT}\n\n"
                "Consider:\n"
                "- Is this already in R&S's known monitoring roadmap?\n"
                "- Does this come from an adjacent field they might not watch?\n"
                "- Could this disrupt their hardware-centric business model?\n"
                "- Is there a non-obvious second-order effect?\n"
                "- For speculative (unvalidated) hypotheses: is the IDEA interesting even without evidence?"
            )},
            {"role": "user", "content": (
                f"HYPOTHESIS ({hv.hypothesis_id}): {hv.hypothesis}\n\n"
                f"STATUS: {hv.verdict.upper()}\n"
                f"{evidence_note}\n"
                f"Summary: {hv.evidence_summary}"
            )},
        ],
        response_format=SurpriseAssessment,
        temperature=0.4,
    )

    sa = response.choices[0].message.parsed
    surprise_results[hv.hypothesis_id] = sa
    tag = "BLIND SPOT" if sa.surprise_rating >= 4 else ""
    print(f"  {hv.hypothesis_id} [{hv.verdict:>11}]: surprise {sa.surprise_rating}/5 {tag}")
    time.sleep(0.5)

# Attach surprise to verdicts
for hv in validated + speculative:
    hv.surprise = surprise_results.get(hv.hypothesis_id)

blind_spots = sum(1 for sa in surprise_results.values() if sa.surprise_rating >= 4)
print(f"\nBlind spots (surprise >= 4): {blind_spots}/{len(surprise_results)}")

## 7. Results Dashboard

In [ ]:
all_verdicts = validated + speculative + falsified

rows = []
for hv in all_verdicts:
    sa = surprise_results.get(hv.hypothesis_id)
    h = next(h for h in all_hypotheses if h.id == hv.hypothesis_id)
    rows.append({
        "ID": hv.hypothesis_id,
        "Framework": h.disruption_framework,
        "Hypothesis": hv.hypothesis[:90],
        "Verdict": hv.verdict,
        "Papers": hv.papers_found,
        "Faithfulness": f"{hv.faithfulness_result.faithfulness_score:.0%}" if hv.faithfulness_result else "—",
        "Surprise": f"{sa.surprise_rating}/5" if sa else "—",
        "Known by R&S": sa.known_by_rs if sa else None,
    })

results_df = pd.DataFrame(rows)
results_df.sort_values(["Verdict", "Surprise"], ascending=[True, False])

In [ ]:
# Verdict distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Verdict counts by framework
verdict_fw = results_df.groupby(["Framework", "Verdict"]).size().unstack(fill_value=0)
colors = {"validated": "#2ecc71", "speculative": "#f39c12", "falsified": "#e74c3c"}
verdict_fw.plot(kind="barh", stacked=True, ax=axes[0],
                color=[colors.get(c, "grey") for c in verdict_fw.columns])
axes[0].set_title("Verdict by Framework")
axes[0].set_xlabel("Count")

# 2. Surprise distribution by verdict
for verdict_type, color in colors.items():
    subset = [surprise_results[hv.hypothesis_id].surprise_rating
              for hv in all_verdicts
              if hv.verdict == verdict_type and hv.hypothesis_id in surprise_results]
    if subset:
        axes[1].hist(subset, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
                     alpha=0.7, label=verdict_type, color=color)
axes[1].set_title("Surprise Distribution by Verdict")
axes[1].set_xlabel("Surprise Rating")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].set_xticks([1, 2, 3, 4, 5])

# 3. Papers found vs surprise
for hv in all_verdicts:
    sa = surprise_results.get(hv.hypothesis_id)
    if sa:
        c = colors.get(hv.verdict, "grey")
        axes[2].scatter(hv.papers_found, sa.surprise_rating, c=c, s=80,
                       edgecolors="black", linewidths=0.5, alpha=0.8)
axes[2].set_title("Evidence vs. Surprise")
axes[2].set_xlabel("Papers Found")
axes[2].set_ylabel("Surprise Rating")
axes[2].set_yticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.show()

## 8. Audit Cards — Full Trail per Hypothesis

In [ ]:
from IPython.display import Markdown, display

# Sort by surprise (highest first), then by verdict (speculative first — most interesting)
verdict_order = {"speculative": 0, "validated": 1, "falsified": 2}
sorted_verdicts = sorted(
    all_verdicts,
    key=lambda hv: (
        -(surprise_results[hv.hypothesis_id].surprise_rating if hv.hypothesis_id in surprise_results else 0),
        verdict_order.get(hv.verdict, 3),
    ),
)

for hv in sorted_verdicts[:15]:
    h = next(h for h in all_hypotheses if h.id == hv.hypothesis_id)
    sa = surprise_results.get(hv.hypothesis_id)
    papers = hypothesis_evidence[h.id]

    verdict_emoji = {"validated": "VALIDATED", "speculative": "SPECULATIVE", "falsified": "FALSIFIED"}
    verdict_label = verdict_emoji.get(hv.verdict, hv.verdict)

    # Evidence section
    if papers:
        papers_md = "\n".join(
            f"  - *{p['title'][:80]}* ({p['year']}, cited: {p['cited_by_count']})"
            for p in papers[:5]
        )
    else:
        papers_md = "  - No academic evidence found"

    # Faithfulness section
    if hv.faithfulness_result:
        faith_md = f"**Faithfulness:** {hv.faithfulness_result.faithfulness_score:.0%}"
        unsupported = [c for c in hv.faithfulness_result.claims if not c.supported]
        if unsupported:
            faith_md += "\n\n**Unsupported claims:**\n" + "\n".join(
                f"  - {c.claim}" for c in unsupported[:3]
            )
    else:
        faith_md = "**Faithfulness:** N/A (no evidence to check against)"

    # Surprise section
    surprise_md = ""
    if sa:
        surprise_md = (
            f"**Surprise:** {sa.surprise_rating}/5 | "
            f"**Known by R&S:** {'Yes' if sa.known_by_rs else 'No'}\n\n"
            f"{sa.explanation}\n\n"
            f"**R&S Relevance:** {sa.rs_relevance}"
        )
        if sa.contrarian_angle:
            surprise_md += f"\n\n**Contrarian angle:** {sa.contrarian_angle}"

    card = f"""### {hv.hypothesis_id}: {h.hypothesis[:100]}

**Verdict:** {verdict_label} | **Framework:** {h.disruption_framework}

{hv.evidence_summary}

**Search queries used:** {', '.join(f'"{q}"' for q in h.search_queries)}

**Papers found ({len(papers)}):**
{papers_md}

{faith_md}

{surprise_md}

---"""
    display(Markdown(card))

## 9. Export Audit Report

In [ ]:
output_dir = Path("../data/audit_reports")
output_dir.mkdir(parents=True, exist_ok=True)

report = {
    "domain": "Regulatory Spectrum Monitoring",
    "approach": "hypothesis-first (poc2d)",
    "n_hypotheses": len(all_hypotheses),
    "n_validated": len(validated),
    "n_speculative": len(speculative),
    "n_falsified": len(falsified),
    "hypotheses": [h.model_dump() for h in all_hypotheses],
    "verdicts": [hv.model_dump() for hv in all_verdicts],
    "surprise_assessments": {k: v.model_dump() for k, v in surprise_results.items()},
    "evidence_search": {
        h_id: [{"title": p["title"], "openalex_id": p["openalex_id"], "year": p["year"],
                "query": p["query"]} for p in papers]
        for h_id, papers in hypothesis_evidence.items()
    },
}

with open(output_dir / "poc2d_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Saved to {output_dir / 'poc2d_report.json'}")
print(f"Report size: {(output_dir / 'poc2d_report.json').stat().st_size / 1024:.0f} KB")

## 10. Comparison: poc2c (Bottom-Up) vs. poc2d (Top-Down)

Load poc2c results and compare: what did each approach find that the other didn't?

In [ ]:
poc2c_path = output_dir / "poc2c_audit_reports.json"

if poc2c_path.exists():
    with open(poc2c_path) as f:
        poc2c_reports = json.load(f)

    poc2c_drivers = [r["driver"]["name"] for r in poc2c_reports]
    poc2c_surprise = {r["driver"]["name"]: r["surprise_llm"]["surprise_rating"] for r in poc2c_reports}
    poc2c_known = {r["driver"]["name"]: r["surprise_llm"]["known_by_rs"] for r in poc2c_reports}

    print("=== poc2c (Bottom-Up) ===")
    print(f"Drivers: {len(poc2c_drivers)}")
    print(f"Surprise >= 4: {sum(1 for s in poc2c_surprise.values() if s >= 4)}")
    print(f"Known by R&S: {sum(1 for k in poc2c_known.values() if k)}/{len(poc2c_known)}")
    print(f"\nDrivers: {', '.join(poc2c_drivers[:10])}{'...' if len(poc2c_drivers) > 10 else ''}")

    print("\n=== poc2d (Top-Down) ===")
    print(f"Hypotheses: {len(all_hypotheses)}")
    print(f"Validated: {len(validated)}, Speculative: {len(speculative)}, Falsified: {len(falsified)}")
    poc2d_surprise_vals = [sa.surprise_rating for sa in surprise_results.values()]
    print(f"Surprise >= 4: {sum(1 for s in poc2d_surprise_vals if s >= 4)}")
    poc2d_known = sum(1 for sa in surprise_results.values() if sa.known_by_rs)
    print(f"Known by R&S: {poc2d_known}/{len(surprise_results)}")

    print("\n=== Key Question ===")
    print("Did poc2d find hypotheses that poc2c could NOT have found?")
    print(f"Speculative hypotheses (no academic evidence): {len(speculative)}")
    print("These are the candidates for genuine blind spots — cross-domain ideas")
    print("that don't appear in the citation graph poc2c traversed.")
else:
    print("poc2c report not found — skip comparison")